# BT-03 GPU Worker (Google Colab)
Sinh ảnh Art Deco 512x512 bằng GPU (T4 free), đẩy sang VM4 (100.70.241.87).

**Chạy tuần tự các cell.** Cần điền `VM4_IP`, `DB_PASS`, `SSH_PASS` ở Cell 2.

In [ ]:
# Cell 1 - Cài đặt dependencies
!pip -q install diffusers==0.30.0 transformers accelerate safetensors \
    torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install psycopg2-binary paramiko pillow pyyaml
print('deps installed')

In [ ]:
# Cell 2 - Cấu hình
VM4_IP = '100.70.241.87'          # Tailscale IP của VM4
VM4_SSH_USER = 'root'
VM4_SSH_PASS = '070995'
DB_NAME = 'bt03'
DB_USER = 'bt03'
DB_PASS = 'B4Deco_Pg_2026!'
WORKER_CODE = '6.1'                # worker GPU (không trùng với 1.1-5.1)
MODEL_ID = 'runwayml/stable-diffusion-v1-5'
NUM_IMAGES = 10                    # số ảnh sinh trong lần chạy
CONCEPTS = ['golden peacock','vintage car','egyptian pharaoh','celestial sun',
            'luxury train','phoenix','art deco skyscraper','diamond geometry',
            'golden butterfly','art deco woman']
print('config ok')

In [ ]:
# Cell 3 - Load model GPU
import torch
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, safety_checker=None)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()
print('model loaded on', torch.cuda.get_device_name(0))

In [ ]:
# Cell 4 - Post-process Art Deco (palette 6 màu + viền vàng 3px + title plate)
import io, hashlib, uuid
from PIL import Image, ImageDraw, ImageFont, ImageFilter

PALETTE = [(26,26,26),(201,162,39),(244,234,213),(30,110,107),(138,109,31),(232,213,163)]

def _quantize(im):
    pal = Image.new('P',(1,1)); flat=[]
    for c in PALETTE: flat.extend(c)
    flat.extend([0]*(768-len(flat)))
    pal.putpalette(flat)
    return im.convert('RGB').filter(ImageFilter.GaussianBlur(0.6)).quantize(palette=pal,dither=Image.NONE).convert('RGB')

def _frame(im, w=3):
    W,H = im.size
    out = Image.new('RGB',(W+w*2,H+w*2),PALETTE[1])
    out.paste(im,(w,w))
    return out

def _title(im, text, band=40):
    W,H = im.size; out=im.copy(); d=ImageDraw.Draw(out)
    d.rectangle((0,H-band,W,H),fill=PALETTE[0])
    d.line((0,H-band,W,H-band),fill=PALETTE[1],width=1)
    label=' '.join(text.replace('-',' ').split()).upper()
    try: font=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf',16)
    except: font=ImageFont.load_default()
    tw=d.textlength(label,font=font)
    d.text(((W-tw)/2,H-band+(band-16)/2),label,font=font,fill=PALETTE[5])
    return out

def postprocess(img_bytes):
    from PIL import Image as I
    im = I.open(io.BytesIO(img_bytes)).convert('RGB').resize((512,512),I.LANCZOS)
    im = _frame(_title(_quantize(im), 'art deco'))
    buf=io.BytesIO(); im.save(buf,format='PNG',optimize=True)
    return buf.getvalue(), im.size
print('postprocess ready')

In [ ]:
# Cell 5 - Sinh ảnh (batch)
import torch
POSITIVE = ('Premium professional illustration of {c}, subject immediately recognizable, '
            'natural proportions, rich details and textures, vibrant harmonious colors, '
            'beautiful Art Deco poster design, elegant geometric ornaments, subtle retro pop art accents, '
            'refined lighting, crisp detailed artwork, sophisticated composition, strong focal point, '
            'polished commercial illustration, clean complementary background, no text, no watermark')
NEGATIVE = ('abstract subject, unrecognizable object, wrong object, incorrect shape, deformed structure, '
            'unrealistic proportions, incomplete object, cropped subject, multiple main subjects, '
            'duplicate objects, overlapping objects, busy background, excessive ornaments, '
            'blurry, pixelated, low resolution, noisy, muddy colors, oversaturated, washed out, '
            'text, typography, letters, watermark, signature')

generated = []
for concept in CONCEPTS[:NUM_IMAGES]:
    slug = concept.lower().replace(' ','-')
    print('generating:', concept)
    with torch.no_grad():
        out = pipe(POSITIVE.format(c=concept), negative_prompt=NEGATIVE,
                   num_inference_steps=25, guidance_scale=7.0,
                   width=512, height=512).images[0]
    buf=io.BytesIO(); out.save(buf,format='PNG'); raw=buf.getvalue()
    img_bytes, size = postprocess(raw)
    sha = hashlib.sha256(img_bytes).hexdigest()
    generated.append((concept, slug, img_bytes, sha, size))
    print('  done', size, len(img_bytes), 'bytes')
print('total generated:', len(generated))

In [ ]:
# Cell 6 - Đẩy sang VM4 (SFTP + PostgreSQL)
import paramiko, psycopg2, uuid
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(VM4_IP, username=VM4_SSH_USER, password=VM4_SSH_PASS, timeout=15)
sftp = ssh.open_sftp()
conn = psycopg2.connect(host=VM4_IP, port=5432, dbname=DB_NAME,
                        user=DB_USER, password=DB_PASS)
cur = conn.cursor()
for concept, slug, img_bytes, sha, size in generated:
    key = f'{slug}/{WORKER_CODE}/{uuid.uuid4().hex}.png'
    remote_dir = f'/var/lib/bt03/images/{slug}/{WORKER_CODE}'
    ssh.exec_command(f'mkdir -p {remote_dir}')
    remote_path = f'{remote_dir}/{key.split(chr(47))[-1]}'
    with sftp.open(remote_path, 'wb') as f:
        f.write(img_bytes)
    cur.execute('''INSERT INTO images (image_id,image_name,slug,prompt,style_id,worker_id,
                   storage_key,sha256,width,height,format,file_size)
                   VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                   ON CONFLICT (sha256) DO NOTHING''',
                (str(uuid.uuid4()), f'{slug} {WORKER_CODE}.png', slug, concept,
                 'deco-v1', 6, key, sha, size[0], size[1], 'png', len(img_bytes)))
    print('pushed', key)
conn.commit(); cur.close(); conn.close(); sftp.close(); ssh.close()
print('ALL DONE -> check http://'+VM4_IP+'/')

## Ghi chú
- Colab free T4: ~25s/ảnh ở 512x512, 25 steps.
- Worker code `6.1` (không trùng 1.1-5.1). Nếu muốn chạy liên tục, bọc Cell 5-6 trong vòng lặp polling Redis.
- Ảnh sinh ra sẽ xuất hiện trong gallery VM4 (hard refresh `Ctrl+F5`).
- Nếu SSH fail, kiểm tra VM4 firewall / Tailscale.